In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error , mean_squared_error , r2_score
from scipy import stats

sns.set_style("whitegrid")
pd.set_option("display.max_columns",50)
pd.set_option("display.float_format","{:.2f}".format)

print("Libraries loaded.")

Libraries loaded.


In [22]:
# 1. CSV read karein (r ke sath bina space ke)
df_raw = pd.read_csv(r"C:\Users\HP\Downloads\bikes.csv")

# 2. Data ko screen par dekhne ke liye
df_raw.head()

,model_name,model_year,kms_driven,owner,location,mileage,power,price
0,Bajaj Avenger Cruise 220 2017,2017,17000 Km,first owner,hyderabad,\n\n 35 kmpl,19 bhp,63500
1,Royal Enfield Classic 350cc 2016,2016,50000 Km,first owner,hyderabad,\n\n 35 kmpl,19.80 bhp,115000
2,Hyosung GT250R 2012,2012,14795 Km,first owner,hyderabad,\n\n 30 kmpl,28 bhp,300000
3,Bajaj Dominar 400 ABS 2017,2017,Mileage 28 Kms,first owner,pondicherry,\n\n 28 Kms,34.50 bhp,100000
4,Jawa Perak 330cc 2020,2020,2000 Km,first owner,bangalore,\n\n,30 bhp,197500


In [23]:
df_raw['owner'].unique()

<StringArray>
['first owner', 'third owner', 'second owner', 'fourth owner or more']
Length: 4, dtype: str

In [24]:
df_raw.dtypes

model_name      str
model_year    int64
kms_driven      str
owner           str
location        str
mileage         str
power           str
price         int64
dtype: object

In [25]:
df_raw.duplicated().sum

<bound method Series.sum of 0       False
1       False
2       False
3       False
4       False
        ...  
7852    False
7853    False
7854    False
7855    False
7856    False
Length: 7857, dtype: bool>

In [26]:
df_raw.isnull().sum()

model_name     0
model_year     0
kms_driven     0
owner          0
location      19
mileage       11
power         31
price          0
dtype: int64

In [27]:
df_raw['price'].describe()

count      7857.00
mean     106791.34
std      138926.12
min           0.00
25%       42000.00
50%       75000.00
75%      125000.00
max     3000000.00
Name: price, dtype: float64

In [28]:
df_raw[['kms_driven','mileage','power']]

,kms_driven,mileage,power
0,17000 Km,\n\n 35 kmpl,19 bhp
1,50000 Km,\n\n 35 kmpl,19.80 bhp
2,14795 Km,\n\n 30 kmpl,28 bhp
3,Mileage 28 Kms,\n\n 28 Kms,34.50 bhp
4,2000 Km,\n\n,30 bhp
...,...,...,...
7852,7000 Km,\n\n 42 kmpl,16 bhp
7853,Mileage 80 Kmpl,\n\n 80 Kmpl,7.7
7854,6407 Km,\n\n 65 kmpl,17 bhp
7855,7524 Km,\n\n 57 kmpl,11.80 bhp


In [29]:
def get_number(value):
    if pd.isna(value,):
        return np.nan
    value=value.replace(',','')
    number = ''
    dot_used = False

    for i in value:
        if i.isdigit():
            number+=i
        elif i == '.' and not dot_used:
            dot_used = True
        elif number:
            break
    return float(number) if number else np.nan

In [30]:
column_name = ['mileage', 'power', 'kms_driven']
for i in column_name:
    df_raw[i] = df_raw[i].apply(get_number)

In [31]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7857 entries, 0 to 7856
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   model_name  7857 non-null   str    
 1   model_year  7857 non-null   int64  
 2   kms_driven  7854 non-null   float64
 3   owner       7857 non-null   str    
 4   location    7838 non-null   str    
 5   mileage     7083 non-null   float64
 6   power       7826 non-null   float64
 7   price       7857 non-null   int64  
dtypes: float64(3), int64(2), str(3)
memory usage: 491.2 KB


In [32]:
df_raw.duplicated().sum()

np.int64(0)

In [33]:
df = df_raw.dropna(subset='price')


In [34]:
df

,model_name,model_year,kms_driven,owner,location,mileage,power,price
0,Bajaj Avenger Cruise 220 2017,2017,17000.00,first owner,hyderabad,35.00,19.00,63500
1,Royal Enfield Classic 350cc 2016,2016,50000.00,first owner,hyderabad,35.00,1980.00,115000
2,Hyosung GT250R 2012,2012,14795.00,first owner,hyderabad,30.00,28.00,300000
3,Bajaj Dominar 400 ABS 2017,2017,28.00,first owner,pondicherry,28.00,3450.00,100000
4,Jawa Perak 330cc 2020,2020,2000.00,first owner,bangalore,NaN,30.00,197500
...,...,...,...,...,...,...,...,...
7852,Yamaha YZF-R15 150cc 2011,2011,7000.00,first owner,agra,42.00,16.00,55000
7853,Bajaj Discover 100cc 2015,2015,80.00,first owner,delhi,80.00,77.00,28000
7854,Bajaj Pulsar 180cc 2016,2016,6407.00,first owner,bangalore,65.00,17.00,61740
7855,Bajaj V15 150cc 2016,2016,7524.00,first owner,bangalore,57.00,1180.00,49000


In [35]:
df[df['price']>0]

,model_name,model_year,kms_driven,owner,location,mileage,power,price
0,Bajaj Avenger Cruise 220 2017,2017,17000.00,first owner,hyderabad,35.00,19.00,63500
1,Royal Enfield Classic 350cc 2016,2016,50000.00,first owner,hyderabad,35.00,1980.00,115000
2,Hyosung GT250R 2012,2012,14795.00,first owner,hyderabad,30.00,28.00,300000
3,Bajaj Dominar 400 ABS 2017,2017,28.00,first owner,pondicherry,28.00,3450.00,100000
4,Jawa Perak 330cc 2020,2020,2000.00,first owner,bangalore,NaN,30.00,197500
...,...,...,...,...,...,...,...,...
7852,Yamaha YZF-R15 150cc 2011,2011,7000.00,first owner,agra,42.00,16.00,55000
7853,Bajaj Discover 100cc 2015,2015,80.00,first owner,delhi,80.00,77.00,28000
7854,Bajaj Pulsar 180cc 2016,2016,6407.00,first owner,bangalore,65.00,17.00,61740
7855,Bajaj V15 150cc 2016,2016,7524.00,first owner,bangalore,57.00,1180.00,49000


In [39]:
df = df[df['price']>0]

In [46]:
pd.Timestamp.today().year

2026

In [47]:
if 'model_year' in df.columns:
    current_year = pd.Timestamp.today().year
    df['model_year']=df['model_year'].between(1998,current_year)